In [0]:
%run /Workspace/Allianz/Allianz_COE/ETL_Ingestion/utils/utils_nb

In [0]:
dbutils.widgets.dropdown(
    name="run_mode",
    defaultValue="ETL",
    choices=["ETL", "Manual"],
    label="Run Mode (ETL / Manual)"
)

dbutils.widgets.text(
    name="input_file_name",
    defaultValue="all",
    label="file name to process"
)

dbutils.widgets.text(
    name="catalog_name",
    defaultValue="allianz_coe",
    label="Control catalog name"
)

dbutils.widgets.text(
    name="control_schema_name",
    defaultValue="audit_control",
    label="Audit Schema Name"
)

dbutils.widgets.text(
    name="brz_metadata_table",
    defaultValue="brz_ingest_metadata",
    label="Ingest metadata table"
)


In [0]:
run_mode = dbutils.widgets.get("run_mode").strip().lower()
selected_table = dbutils.widgets.get("input_file_name").strip().lower()
catalog_name = dbutils.widgets.get("catalog_name").strip().lower()
aud_schema_name = dbutils.widgets.get("control_schema_name").strip().lower()
metadata_table = dbutils.widgets.get("brz_metadata_table").strip().lower()

In [0]:
def build_source_path(file_path_or_table: str, file_or_table_name: str) -> str:
    return f"{file_path_or_table.rstrip('/')}/{file_or_table_name}"

In [0]:
from pyspark.sql.types import StructType
import json

def get_spark_reader(schema_mode: str, schema_json: str,task_run_id: str):
    reader = spark.read

    if schema_mode.lower() == "explicit":
        if not schema_json:
            task_log(task_run_id=task_run_id, task_name="get_spark_reader", message="schema_json must be provided when schema_mode is explicit")
            raise ValueError("schema_json must be provided when schema_mode is explicit")

        schema = StructType.fromJson(json.loads(schema_json))
        reader = reader.schema(schema)

    elif schema_mode.lower() == "infer":
        reader = reader.option("inferSchema", "true")

    elif schema_mode.lower() == "string":
        reader = reader

    else:
        raise ValueError(f"Unsupported schema_mode: {schema_mode}")

    return reader

In [0]:
def read_input_file(
    reader,
    source_path: str,
    file_type: str,
    header: bool,
    task_run_id: str
):
    file_type = file_type.lower()
    ##task_log(task_run_id=task_run_id, task_name="read_input_file", message=f"file type is {file_type}")
    if file_type == "csv":
        df = reader.format("csv")\
                  .option("header", header)\
                  .load(source_path)
        
        return (
            df
        )

    elif file_type == "json":
        
        return reader.format("json").load(source_path)

    else:
        task_log(task_run_id=task_run_id, task_name="read_input_file", message=f"Unsupported file_type: {file_type}")
        raise ValueError(f"Unsupported file_type: {file_type}")

In [0]:
from pyspark.sql.functions import current_timestamp, current_user
from pyspark.sql.functions import lit

def add_audit_columns(df,task_run_id):
    return (
        df.withColumn("load_ts", current_timestamp().cast("string"))
          .withColumn("created_by", lit("ETL System"))
          .withColumn("created_ts", current_timestamp().cast("string"))
    )

In [0]:
def write_to_bronze_delta(
    df,
    full_table_name: str,
    write_mode: str,
    task_run_id: str
):
    ##task_log(task_run_id=task_run_id, task_name="write_to_delta", message=f"Writing {full_table_name} using {write_mode}")
    (
        df.write
          .format("delta")
          .mode(write_mode.lower())
          .option(
              "overwriteSchema",
              "true" if write_mode.lower() == "overwrite" else "false"
          )
          .saveAsTable(full_table_name)
    )
    task_log(task_run_id=task_run_id, task_name="write_to_bronze_delta", message=f"{full_table_name} is successful")

In [0]:
def process_metadata_row(row,task_run_id):

    file_path      = row["landing_path"]
    file_name      = row["object_name"]
    file_format    = row["file_format"]
    header_flag    = row["header_flag"]
    schema_mode    = row["schema_mode"]
    schema_json    = row["schema_json"]
    write_mode     = row["write_mode"]
    target_catalog = row["target_catalog"]
    target_schema  = row["target_schema"]
    target_table   = row["target_table"]
    load_type      = row["load_type"]
    delimiter      = row["delimiter"]
    multiline_flag = row["multiline_flag"]
    encoding       = row["encoding"]

    if load_type.lower() == "incremental":
        
        landing = dbutils.fs.ls(file_path)
        folders = [f.path for f in landing if f.isDir() and "archive" not in f.path.lower()]
        latest_folder = sorted(folders,reverse=True)[0]
        print(f"Picking file from latest folder {latest_folder}")
        source_path = build_source_path(
            latest_folder, file_name
        )
        ##task_log(task_run_id=task_run_id, task_name="Source Path", message=f"{source_path}")
        full_table_name = f"{target_catalog}.{target_schema}.{target_table}"

        ##task_log(task_run_id=task_run_id, task_name="Get Reader", message=f"Reader for schema {schema_mode}")
        reader = get_spark_reader(schema_mode, schema_json,task_run_id = task_run_id)

        task_log(task_run_id=task_run_id, task_name="read_input_file", message=f"Reading input file {file_name}")

        df = read_input_file(
            reader=reader,
            source_path=source_path,
            file_type=file_format,
            header=header_flag,
            task_run_id = task_run_id
        )

        ##task_log(task_run_id=task_run_id, task_name="add_audit_columns", message="Adding audit columns")
        # Add audit columns
        df = add_audit_columns(df,task_run_id = task_run_id)

        task_log(task_run_id=task_run_id, task_name="write_to_bronze_delta", message=f"Writing to delta table {full_table_name}")
        write_to_bronze_delta(
            df=df,
            full_table_name=full_table_name,
            write_mode=write_mode,
            task_run_id = task_run_id
        )
        print(f"Processed {file_name} and is available at {full_table_name}")
    else:
        task_log(task_run_id=task_run_id, task_name="load_type", message=f"Unsupported load_type: {load_type}. load_type should be incremental")
        raise ValueError(f"Unsupported load_type: {load_type}. load_type should be incremental")

In [0]:
def run_metadata_ingestion(
    metadata_table: str,
    run_mode: str,
    selected_table: str,
    task_run_id: str
):
    task_log(task_run_id=task_run_id, task_name="run_metadata_ingestion", message="Start Running run_metadata_ingestion function")
    
    metadata_df = (
        spark.table(metadata_table)
             .filter("is_active = true")
    )
    task_log(task_run_id=task_run_id, task_name="metadata table creation", message="Active tables added to metadata_df")
    task_log(task_run_id=task_run_id, task_name="run_mode", message=f"{run_mode}")
    # Manual mode filtering
    if run_mode == "manual":
        if selected_table and selected_table != "all":
            metadata_df = metadata_df.filter(
                f"lower(file_name) = '{selected_table}'"
            )
            task_log(task_run_id=task_run_id, task_name="Manual Run Table", message=f"{selected_table}")

    if metadata_df.count() == 0:
        task_log(task_run_id=task_run_id, task_name="Metadata records", message="No metadata records found to process")
        raise ValueError("No metadata records found to process")

    for row in metadata_df.collect():
        task_log(task_run_id=task_run_id, task_name="Metadata tables available", message="Processing metadata row")
        process_metadata_row(row,task_run_id = task_run_id)

In [0]:
METADATA_TABLE = f"{catalog_name}.{aud_schema_name}.{metadata_table}" # Needs to be widget param

log_message = 'Reading Metadata Tables & Parameters'
task_run_id = '899'
current_task_name = "Running run_metadata_ingestion"

task_log(task_run_id=task_run_id, task_name=current_task_name, message=f"{log_message}")

run_metadata_ingestion(
    metadata_table=METADATA_TABLE,
    run_mode=run_mode,
    selected_table=selected_table,
    task_run_id = task_run_id
)

task_log(task_run_id=task_run_id, task_name="Landing_to_Bronze", message="The landing to bronze is completed")

# Msg taks_log that landing to bronze is completed

Picking file from latest folder dbfs:/Volumes/allianz_coe/bronze/files/source_1/csv/20260407_174341_42/
Processed crm_account.csv and is available at allianz_coe.amit_raw_test.crm_account
Picking file from latest folder dbfs:/Volumes/allianz_coe/bronze/files/source_1/csv/20260407_174341_42/
Processed crm_consent.csv and is available at allianz_coe.amit_raw_test.crm_consent
Picking file from latest folder dbfs:/Volumes/allianz_coe/bronze/files/source_1/csv/20260407_174341_42/
Processed crm_contact.csv and is available at allianz_coe.amit_raw_test.crm_contact
Picking file from latest folder dbfs:/Volumes/allianz_coe/bronze/files/source_1/csv/20260407_174341_42/
Processed crm_customer.csv and is available at allianz_coe.amit_raw_test.crm_customer
Picking file from latest folder dbfs:/Volumes/allianz_coe/bronze/files/source_1/csv/20260407_174341_42/
Processed crm_customer_lead.csv and is available at allianz_coe.amit_raw_test.crm_customer_lead
Picking file from latest folder dbfs:/Volumes/